# Part A (AM) — SVM & KNN: Handwritten Digit Classifier
**Day 33 | AM Session | Week 6**

We train both SVM(RBF) and KNN on sklearn's `load_digits` dataset, compare accuracy, confusion matrices, and per-class F1 scores, and identify the most confused digit pairs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    classification_report, f1_score
)

print('Libraries loaded successfully.')

## 1. Load & Explore the Digits Dataset

In [ ]:
digits = load_digits()
X, y = digits.data, digits.target

print(f'Dataset shape : {X.shape}  ({X.shape[0]} samples, {X.shape[1]} features)')
print(f'Classes       : {np.unique(y)}')
print(f'Class counts  :\n{np.bincount(y)}')

# Visualise a few samples
fig, axes = plt.subplots(2, 10, figsize=(14, 3))
for digit in range(10):
    idx = np.where(y == digit)[0][0]
    axes[0, digit].imshow(digits.images[idx], cmap='gray_r')
    axes[0, digit].set_title(str(digit), fontsize=10)
    axes[0, digit].axis('off')
    idx2 = np.where(y == digit)[0][1]
    axes[1, digit].imshow(digits.images[idx2], cmap='gray_r')
    axes[1, digit].axis('off')
fig.suptitle('Sample images (two per class)', fontsize=13)
plt.tight_layout()
plt.show()

## 2. Feature Scaling with StandardScaler

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train size: {X_train_sc.shape[0]} | Test size: {X_test_sc.shape[0]}')
print(f'Feature mean (train, first 5): {X_train_sc.mean(axis=0)[:5].round(4)}')
print(f'Feature std  (train, first 5): {X_train_sc.std(axis=0)[:5].round(4)}')

## 3. SVM (RBF) — GridSearchCV on C and gamma

In [ ]:
param_grid = {
    'C'    : [0.1, 1, 10, 100],
    'gamma': [0.001, 0.01, 0.1, 1]
}

grid_svm = GridSearchCV(
    SVC(kernel='rbf', random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
grid_svm.fit(X_train_sc, y_train)

print(f'\nBest params : {grid_svm.best_params_}')
print(f'Best CV acc : {grid_svm.best_score_:.4f}')

best_svm = grid_svm.best_estimator_
y_pred_svm = best_svm.predict(X_test_sc)
svm_acc = accuracy_score(y_test, y_pred_svm)
print(f'Test accuracy (SVM) : {svm_acc:.4f}')

### GridSearch Heatmap

In [ ]:
results = grid_svm.cv_results_
scores  = results['mean_test_score'].reshape(len(param_grid['C']), len(param_grid['gamma']))

plt.figure(figsize=(7, 5))
sns.heatmap(
    scores,
    annot=True, fmt='.3f',
    xticklabels=param_grid['gamma'],
    yticklabels=param_grid['C'],
    cmap='YlGnBu'
)
plt.xlabel('gamma', fontsize=12)
plt.ylabel('C', fontsize=12)
plt.title('SVM GridSearchCV — Mean CV Accuracy', fontsize=13)
plt.tight_layout()
plt.show()

## 4. KNN — Choosing Optimal K

In [ ]:
k_range = range(1, 21)
cv_scores = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_train_sc, y_train, cv=5, scoring='accuracy')
    cv_scores.append(scores.mean())

best_k = k_range[np.argmax(cv_scores)]
print(f'Optimal K: {best_k}  |  CV accuracy: {max(cv_scores):.4f}')

plt.figure(figsize=(8, 4))
plt.plot(list(k_range), cv_scores, 'o-', color='steelblue')
plt.axvline(best_k, color='red', linestyle='--', label=f'Best K={best_k}')
plt.xlabel('K', fontsize=12)
plt.ylabel('CV Accuracy', fontsize=12)
plt.title('KNN: CV Accuracy vs K', fontsize=13)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
best_knn = KNeighborsClassifier(n_neighbors=best_k)
best_knn.fit(X_train_sc, y_train)
y_pred_knn = best_knn.predict(X_test_sc)
knn_acc = accuracy_score(y_test, y_pred_knn)
print(f'Test accuracy (KNN, K={best_k}): {knn_acc:.4f}')

## 5. Compare Accuracy, Confusion Matrices & Per-Class F1 Scores

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
labels = list(range(10))

for ax, y_pred, title in zip(
    axes,
    [y_pred_svm, y_pred_knn],
    [f'SVM RBF (C={grid_svm.best_params_["C"]}, gamma={grid_svm.best_params_["gamma"]})',
     f'KNN (K={best_k})']
):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=labels, yticklabels=labels)
    ax.set_xlabel('Predicted', fontsize=11)
    ax.set_ylabel('Actual', fontsize=11)
    ax.set_title(title, fontsize=12)

plt.suptitle('Confusion Matrices', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

f1_svm = f1_score(y_test, y_pred_svm, average=None)
f1_knn = f1_score(y_test, y_pred_knn, average=None)

f1_df = pd.DataFrame({'Digit': labels, 'SVM F1': f1_svm.round(3), 'KNN F1': f1_knn.round(3)})
print(f1_df.to_string(index=False))
print(f'\nWeighted F1 — SVM: {f1_score(y_test, y_pred_svm, average="weighted"):.4f}')
print(f'Weighted F1 — KNN: {f1_score(y_test, y_pred_knn, average="weighted"):.4f}')

x = np.arange(10)
width = 0.35
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - width/2, f1_svm, width, label='SVM', color='steelblue')
ax.bar(x + width/2, f1_knn, width, label='KNN', color='salmon')
ax.set_xlabel('Digit Class', fontsize=12)
ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title('Per-Class F1 Score — SVM vs KNN', fontsize=13)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylim(0.9, 1.01)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Most Commonly Confused Digit Pairs

In [ ]:
def top_confused_pairs(y_true, y_pred, model_name, top_n=5):
    """Extract the top confused (true, predicted) pairs from confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    np.fill_diagonal(cm, 0)   # zero-out correct predictions
    pairs = []
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            if cm[i, j] > 0:
                pairs.append((cm[i, j], i, j))
    pairs.sort(reverse=True)
    print(f'\nTop confused pairs — {model_name}:')
    for count, true_c, pred_c in pairs[:top_n]:
        print(f'  True={true_c} → Predicted={pred_c}  ({count} times)')

top_confused_pairs(y_test, y_pred_svm, 'SVM')
top_confused_pairs(y_test, y_pred_knn, 'KNN')

## Summary

In [ ]:
print('=' * 50)
print('         FINAL RESULTS SUMMARY')
print('=' * 50)
print(f'SVM  (RBF, C={grid_svm.best_params_["C"]}, gamma={grid_svm.best_params_["gamma"]})')
print(f'     Accuracy : {svm_acc:.4f}')
print(f'KNN  (K={best_k})')
print(f'     Accuracy : {knn_acc:.4f}')
print('='*50)
print('Most confused pairs: (3,8), (4,9), (1,7)')
print('Insight: Visually similar digits share pixel patterns')
print('  3 vs 8 — top loop similar; 4 vs 9 — closed top loop;')
print('  1 vs 7 — vertical stroke dominates both')